# RuVLM BenchMax — обучение на GPU (Google Colab)

Дообучение `deepvk/llava-gemma-2b-lora` на открытых данных VK (GQA-ru) через LoRA.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU** (или лучше).

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Включите GPU в Runtime → Change runtime type"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# peft новых версий требует torchao>=0.16 (в Colab часто стоит 0.10)
!pip install -q -U "torchao>=0.16.0" "transformers>=4.43" "peft>=0.11" "trl>=0.9" "datasets>=2.20" "accelerate>=0.33" bitsandbytes Pillow PyYAML

# Если Runtime уже импортировал старый peft — перезапустите runtime после этой ячейки
import importlib, sys
for m in list(sys.modules):
    if m == "peft" or m.startswith("peft."):
        del sys.modules[m]
print("deps ready")


## (Опционально) клонировать репозиторий проекта

In [ ]:
# Опционально — актуальный репозиторий проекта:
# !git clone https://github.com/DanilaIsakov/VK_practice.git
# %cd VK_practice


## Быстрый LoRA fine-tune на GQA-ru (subset)

Для полного прогона увеличьте `MAX_SAMPLES` и `NUM_EPOCHS`.

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoProcessor,
    AutoTokenizer,
    LlavaForConditionalGeneration,
    TrainingArguments,
)
from trl import SFTTrainer

MODEL_ID = "deepvk/llava-gemma-2b-lora"
OUTPUT_DIR = "outputs/ruvlm-gemma-2b-lora"
MAX_SAMPLES = 1000  # увеличьте для лучшего качества
NUM_EPOCHS = 1
POST_PROMPT = " Ответь одним словом."

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
processor.tokenizer = tokenizer

lora = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

gqa = load_dataset("deepvk/GQA-ru", split="train")
gqa = gqa.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(gqa))))


def map_row(ex):
    q = ex["question"].rstrip() + POST_PROMPT
    a = ex.get("answer") or ex.get("fullAnswer") or ""
    return {
        "messages": [
            {"role": "user", "content": f"<image>\n{q}"},
            {"role": "assistant", "content": str(a)},
        ],
        "image": ex.get("image"),
    }


train_ds = gqa.map(map_row, remove_columns=gqa.column_names)


class Collator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts, images = [], []
        for ex in examples:
            text = self.processor.tokenizer.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(ex["image"])
        batch = self.processor(text=texts, images=images, return_tensors="pt", padding=True)
        labels = batch["input_ids"].clone()
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        batch["labels"] = labels
        return batch


args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=200,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=Collator(processor),
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

## Демо инференса

In [ ]:
import requests
from PIL import Image

url = "https://www.ilankelman.org/stopsigns/australia.jpg"
img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
messages = [{"role": "user", "content": "<image>\nОпиши картинку несколькими словами."}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(images=[img], text=text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Оценка на бенчмарках

После сохранения чекпоинта установите `lmms-eval` и запустите:

```bash
accelerate launch -m lmms_eval --model llava_hf \
  --model_args pretrained=outputs/ruvlm-gemma-2b-lora \
  --tasks gqa-ru,mmbench_ru_dev --batch_size 1 \
  --output_path ./logs/
```

Запишите метрики в `results/metrics.md` и `docs/MODEL_CARD.md`.